# E07 — Redundância celular com os enlaces GE instalados

O projeto original define os enlaces GE instalados. Neste exemplo, o perfil de Barigui passa a ter duas interfaces: GE e celular público. O dispositivo é conectado, com custo inicial 1.0. O acesso da operadora é uma premissa; não há planejamento de enlaces celulares.

Para demonstrar a perda da saída original, desabilitamos a interface GE de **Torre**, não Barigui. Preservamos todos os outros participantes e somente os enlaces GE selecionados no projeto original. Não são gerados novos enlaces por proximidade.

Este notebook contém apenas chamadas ao SDK e usa visualizações de grafos. Requer SDK 0.1.39+ e serviço atualizado. Execute o notebook 1 primeiro, com o kernel na pasta notebooks.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject, ExtendedPlanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1900.0)
source = PlanningProject("../projects/e07_ge_mesh_original")


## 1. Revisar a topologia instalada

No grafo original, Barigui utiliza Torre como pai e encaminha a comunicação de sua sub-rede. Observe também os outros filhos de Torre: eles não recebem celular automaticamente.

In [ ]:
source.show_network();
source.show_dependencies();


## 2. Preparar o perfil duplo e a condição de falha

A receita `extension_fixed_links.toml` contém a substituição por instância, o custo inicial 1.0, o override que desabilita Torre e o modo `topology.mode = "installed"`.

O SDK herda os enlaces da solução original e exclui somente aqueles incidentes à interface desabilitada. A interface GE de Barigui mantém seu ID e continua ativa; a nova interface celular também fica ativa. Como connected e rank são propriedades do dispositivo, ambas herdam essa condição.

A frequência 0.0 da interface externa é apenas um campo técnico do formato, sem significado de propagação. Não são geradas métricas RF para ela.

A preparação abaixo arquiva versões anteriores do segundo projeto antes de regenerá-lo. O projeto original não é modificado. O nome da receita é novo para permitir sua distribuição sem sobrescrever receitas locais do exemplo anterior.

In [ ]:
project = ExtendedPlanningProject(
    "../projects/e07_ge_mesh_redundancia",
    source=source,
    extension_file="extension_fixed_links.toml",
)
project.prepare(overwrite=True);


## 3. Inspecionar os enlaces disponíveis após a falha

São os enlaces instalados sobreviventes, e não novos candidatos geométricos. Torre aparece em cinza como desabilitada. Barigui mantém as duas interfaces; selecione-o para consultar os detalhes.

In [ ]:
scenario = client.open_project(project, planner="graph")
scenario.build_candidate_edges();
scenario.show_network(kind="candidate");


## 4. Avaliar o atendimento pela saída celular

O cálculo utiliza somente os enlaces instalados sobreviventes. A função de roteamento determina o atendimento nessa topologia restrita, sem criar alternativas GE ou trocar os enlaces instalados. A sub-rede de Barigui permanece com seus pais originais, exceto o próprio Barigui, que agora fornece acesso externo.

Sites de outros ramos que dependiam de Torre podem ficar sem serviço. Magenta indica custo acima do limiar; vermelho indica ausência de caminho. A presença de caminho não equivale à aprovação de qualidade.

In [ ]:
scenario.compute_metrics(geo_base_url="http://features:8080", batch_size=10);
scenario.solve();
scenario.save_result(rank_threshold=6.0, primary_tech="ge_mesh");
project.show_network();
project.show_dependencies();


## 5. Interpretação

Selecione Barigui para inspecionar seus descendentes e as interfaces GE e celular. O exemplo demonstra uma saída externa para uma sub-rede, mantendo a topologia instalada. A indisponibilidade é da interface de Torre; não é uma falha do equipamento Barigui.

A segunda interface celular é preservada no inventário, mas não há uma torre da operadora desenhada no grafo. A origem conectada com custo inicial 1.0 representa esse acesso. Não se demonstra aqui proteção contra falha física do próprio Barigui nem comutação automática de pais GE.

Para escolher outro ponto, altere a regra por instância na receita, após consultar a rede original. As alterações de preparação e validação continuam no SDK, nunca em lógica adicional no notebook.